# HW5: Illinois Professional Licenses Visualization


In [ ]:
import pandas as pd
import altair as alt
# Disable
alt.data_transformers.disable_max_rows()

## Load Data

In [27]:
url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv'
df = pd.read_csv(url)
df.head(5)

,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.columns.tolist()

## Plot 1: Top 10 License Types by Status (Interactive Bar Chart)

This plot shows the count of licenses for the top 10 license types, each row's types' total count seperated by license status.

**Design choices:**
- Encoding: x-axis = count (**quantitative**); y-axis = License Type (**nominal**, sorted by total count)
- Color: encodes License Status using the `tableau10` scheme.
- Data transformation: grouped by License Type and License Status, filtered to top 10 types by total count.

**Interactivity:** Clicking on a license type in the legend highlights only that type across all bars, making it easy to compare a specific license type's status distribution.

In [6]:
# Data transformation: count licenses by type and status
type_status_counts = df.groupby(['License Type', 'License Status']).size().reset_index(name='Count')

# find top 10 license types
top_types = df['License Type'].value_counts().head(10).index.tolist()
type_status_filtered = type_status_counts[type_status_counts['License Type'].isin(top_types)]

print(f'Top 10 license types: {top_types}')
type_status_filtered.head(10)

Top 10 license types: ['DETECTIVE BOARD', 'COSMO', 'DENTAL', 'FUNERAL AND EMBALMER', 'DIETETIC AND NUTRITION', 'DESIGN FIRM', 'MASSAGE LICENSING BD', 'HOME INSPECTOR', 'COMM ASSOC MGR', 'CLIN PSYCHOLOGIST']


,License Type,License Status,Count
11,CLIN PSYCHOLOGIST,ACTIVE,9
12,CLIN PSYCHOLOGIST,CANCELLED,5
13,CLIN PSYCHOLOGIST,INACTIVE,3
14,CLIN PSYCHOLOGIST,NOT RENEWED,7
21,COMM ASSOC MGR,ACTIVE,19
22,COMM ASSOC MGR,INACTIVE,2
23,COMM ASSOC MGR,NOT RENEWED,16
24,COSMO,ACTIVE,940
25,COSMO,CANCELLED,12
26,COSMO,CHANGE OF OWNERSHIP,15


In [19]:
# interactive
selection = alt.selection_point(fields=['License Status'], bind='legend')

chart1 = alt.Chart(type_status_filtered).mark_bar().encode(
    x=alt.X('Count:Q', title='Number of Licenses'),
    y=alt.Y('License Type:N', sort='-x', title='License Type'),
    color=alt.Color('License Status:N',
            scale=alt.Scale(scheme='tableau10'),
            title='License Status (Click me!)'),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=['License Type', 'License Status', 'Count']
).add_params(
    selection
).properties(
    title='Top 10 License Types by Status',
    width=600,
    height=400
)

chart1

alt.Chart(...)

In [ ]:
chart1.save('licenses_by_type.json')

## Plot 2: License Issuance Over Time by Type

This plot shows how the number of licenses issued per year has changed over time for the top 5 license types.

**Design choices:**
- Encoding: x-axis = year (**temporal**), y-axis = count of licenses (**quantitative**), color = License Type (**nominal**)
- Color: uses the `category10` scheme to differentiate license types clearly
- Data transformation: parsed `Original Issue Date` to extract year, filtered to designated dates(1990–2022), grouped by year and top 5 license types

In [23]:
# Data transformation: parse dates and extract year
df['Date Parsed'] = pd.to_datetime(df['Original Issue Date'], format='%m/%d/%Y', errors='coerce')
df['Issue Year'] = df['Date Parsed'].dt.year

# top 5 license types
top5_types = df['License Type'].value_counts().head(5).index
# filter years
df_filtered = df[(df['Issue Year'] >= 1990) & (df['Issue Year'] <= 2022) &
                 (df['License Type'].isin(top5_types))]

# group by year and type
timeline = df_filtered.groupby(['Issue Year', 'License Type']).size().reset_index(name='Count')
timeline['Issue Year'] = pd.to_datetime(timeline['Issue Year'], format='%Y')

print(f'Top 5 license types: {top5_types}')
timeline.head(10)

Top 5 license types: ['DETECTIVE BOARD', 'COSMO', 'DENTAL', 'FUNERAL AND EMBALMER', 'DIETETIC AND NUTRITION']


,Issue Year,License Type,Count
0,1990-01-01,COSMO,40
1,1990-01-01,DENTAL,14
2,1990-01-01,DETECTIVE BOARD,135
3,1991-01-01,COSMO,33
4,1991-01-01,DENTAL,11
5,1991-01-01,DETECTIVE BOARD,125
6,1991-01-01,FUNERAL AND EMBALMER,22
7,1992-01-01,COSMO,24
8,1992-01-01,DENTAL,10
9,1992-01-01,DETECTIVE BOARD,125


In [30]:
chart2 = alt.Chart(timeline).mark_line(point=True).encode(
    x=alt.X('Issue Year:T', title='Year'),
    y=alt.Y('Count:Q', title='Number of Licenses Issued'),
    color=alt.Color('License Type:N',
            scale=alt.Scale(scheme='category10'),
            title='License Type (Do not click!)'),
    tooltip=['Issue Year:T', 'License Type:N', 'Count:Q']
).properties(
    title='License Issuance Over Time by Type (1990-2022)',
    width=800,
    height=400
)

chart2

alt.Chart(...)

In [ ]:
chart2.save('licenses_over_time.json')